# 02 — Causal Graph Visualisation

Inspect the learned SCM structure at different training stages.

In [ ]:
import sys; sys.path.insert(0, '..')
import torch
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
from omegaconf import OmegaConf
from models.se_cvla import SECVLA

cfg = OmegaConf.load('../configs/base.yaml')
model = SECVLA.from_pretrained('../outputs/stage3/best.ckpt', cfg)
model.eval()
print('Model loaded.')

In [ ]:
# Get current hard adjacency matrix
adj = model.scm.get_causal_graph().numpy()  # (d, d)
d = adj.shape[0]
print(f'Causal variables: {d}')
print(f'Active edges: {int(adj.sum())}')

# Build networkx graph
G = nx.DiGraph()
for i in range(d): G.add_node(i, label=f'V{i}')
for i in range(d):
    for j in range(d):
        if adj[i, j] > 0.5:
            G.add_edge(j, i)

print(f'Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}')
print(f'Is DAG: {nx.is_directed_acyclic_graph(G)}')

In [ ]:
# Draw the causal graph
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Left: spring layout
pos = nx.spring_layout(G, seed=42)
nx.draw_networkx(G, pos=pos, ax=axes[0], with_labels=True,
    node_color='#4C72B0', font_color='white', node_size=800,
    edge_color='#DD8452', arrows=True, arrowsize=20, font_size=9)
axes[0].set_title('Learned Causal Graph (Spring Layout)')

# Right: adjacency matrix heatmap
im = axes[1].imshow(adj, cmap='Blues', vmin=0, vmax=1)
plt.colorbar(im, ax=axes[1])
axes[1].set_title('Soft Adjacency Matrix W')
axes[1].set_xlabel('Parent variable'); axes[1].set_ylabel('Child variable')

plt.tight_layout(); plt.show()

In [ ]:
# Degree distribution
in_deg  = [G.in_degree(n)  for n in G.nodes()]
out_deg = [G.out_degree(n) for n in G.nodes()]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(d), in_deg);  axes[0].set_title('In-degree (number of causes)')
axes[1].bar(range(d), out_deg); axes[1].set_title('Out-degree (number of effects)')
for ax in axes: ax.set_xlabel('Variable index')
plt.tight_layout(); plt.show()